# MDD Challenge 2025 - Predict Only From Trained PER-MDD Run

This notebook does **not** train again.

It:
- loads the already trained phoneme CTC model,
- recreates the original train/validation split deterministically,
- rebuilds the PER-MDD reference pool from the training split,
- loads the best PER-MDD config from the previous run if available,
- predicts the public test set in the **exact row order of `public_test_phones.csv`**.

Outputs:
- `results.csv` with one column: `predict`
- `predictions.csv` with two columns: `id,predict`


## 1. Setup

In [ ]:
from pathlib import Path
import re
import math
import random
import wave
import json
import subprocess
import sys
import importlib.util
from collections import Counter, defaultdict

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        print(f'Installing missing package: {package}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

ensure_package('torch')
ensure_package('sklearn', 'scikit-learn')
ensure_package('transformers')
ensure_package('accelerate')

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoFeatureExtractor, Wav2Vec2ForCTC, set_seed

BASELINE_SEED = 42
set_seed(BASELINE_SEED)
random.seed(BASELINE_SEED)
np.random.seed(BASELINE_SEED)
torch.manual_seed(BASELINE_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Locate Data And Previous Run

In [ ]:
def find_training_root():
    candidates = [
        Path('/kaggle/input/mdd-challenge-2025-training-set/MDD-Challenge-2025-training-set'),
        Path('/kaggle/input/mdd-challenge-2025-training-set'),
        Path('/kaggle/input/mdd-challenge-data/MDD-Challenge-2025-training-set'),
        Path('/kaggle/input/mdd-challenge-data'),
        Path('./MDD-Challenge-2025-training-set'),
        Path('.'),
    ]
    for p in candidates:
        if (p / 'metadata' / 'train_phones.csv').exists():
            return p
    for train_csv in Path('.').rglob('metadata/train_phones.csv'):
        return train_csv.parent.parent
    raise FileNotFoundError('Training dataset root not found.')

def find_public_test_root():
    candidates = [
        Path('/kaggle/input/mdd-challenge-2025-public-test/MDD-Challenge-2025-public-test'),
        Path('/kaggle/input/mdd-challenge-2025-public-test'),
        Path('/kaggle/input/mdd-challenge-data/MDD-Challenge-2025-public-test/MDD-Challenge-2025-public-test'),
        Path('/kaggle/input/mdd-challenge-data/MDD-Challenge-2025-public-test'),
        Path('./MDD-Challenge-2025-public-test/MDD-Challenge-2025-public-test'),
        Path('./MDD-Challenge-2025-public-test'),
        Path('.'),
    ]
    for p in candidates:
        if (p / 'metadata' / 'public_test_phones.csv').exists() and (p / 'audio_data' / 'public_test').exists():
            return p
    for phones_csv in Path('.').rglob('metadata/public_test_phones.csv'):
        root = phones_csv.parent.parent
        if (root / 'audio_data' / 'public_test').exists():
            return root
    raise FileNotFoundError('Public test root not found.')

def find_previous_run_dirs():
    out_candidates = []
    if Path('/kaggle/working').exists():
        out_candidates.append(Path('/kaggle/working'))
    out_candidates.extend([Path('./outputs'), Path('.')])

    baseline_candidates = []
    per_mdd_candidates = []
    for out_dir in out_candidates:
        baseline_candidates.extend([
            out_dir / 'phoneme_ctc_baseline_public_test',
            out_dir / 'phoneme_ctc_baseline',
        ])
        per_mdd_candidates.extend([
            out_dir / 'phoneme_ctc_per_mdd_public_test',
            out_dir / 'phoneme_ctc_per_mdd',
        ])

    baseline_run_dir = None
    final_model_dir = None
    for cand in baseline_candidates:
        fm = cand / 'final_model'
        if (fm / 'config.json').exists():
            baseline_run_dir = cand
            final_model_dir = fm
            break
    if final_model_dir is None:
        raise FileNotFoundError('Could not find a trained final_model directory from a previous run.')

    per_mdd_dir = None
    for cand in per_mdd_candidates:
        if cand.exists():
            per_mdd_dir = cand
            break
    if per_mdd_dir is None:
        per_mdd_dir = baseline_run_dir.parent / 'phoneme_ctc_per_mdd_public_test_predict_only'
    per_mdd_dir.mkdir(parents=True, exist_ok=True)
    return baseline_run_dir, final_model_dir, per_mdd_dir

TRAIN_ROOT = find_training_root()
PUBLIC_TEST_ROOT = find_public_test_root()
BASELINE_RUN_DIR, FINAL_MODEL_DIR, PER_MDD_DIR = find_previous_run_dirs()

print('TRAIN_ROOT:', TRAIN_ROOT)
print('PUBLIC_TEST_ROOT:', PUBLIC_TEST_ROOT)
print('BASELINE_RUN_DIR:', BASELINE_RUN_DIR)
print('FINAL_MODEL_DIR:', FINAL_MODEL_DIR)
print('PER_MDD_DIR:', PER_MDD_DIR)


## 3. Load Training Metadata And Recreate Split

In [ ]:
TRAIN_PHONES_CSV = TRAIN_ROOT / 'metadata' / 'train_phones.csv'
TRAIN_AUDIO_DIR = TRAIN_ROOT / 'audio_data' / 'train'

train_phones = pd.read_csv(TRAIN_PHONES_CSV)

def normalize_phone_text(s):
    return ' '.join(str(s).replace('*', '').replace('$', '').split())

def eval_phone_tokens(s):
    return normalize_phone_text(s).split()

def resolve_train_audio_path(rel_path):
    rel = Path(str(rel_path))
    candidates = [TRAIN_ROOT / rel, TRAIN_ROOT / 'audio_data' / 'train' / rel.name, TRAIN_AUDIO_DIR / rel.name]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

def infer_speaker_group(sample_id, rel_path):
    stem = Path(str(rel_path)).stem
    has_long_timestamp_suffix = bool(re.search(r'[_-]\d{10,}$', stem)) or bool(re.search(r'[_-]\d{10,}$', str(sample_id)))
    return 'adult' if has_long_timestamp_suffix else 'child'

train_phones['canonical_norm'] = train_phones['canonical'].map(normalize_phone_text)
train_phones['transcript_norm'] = train_phones['transcript'].map(normalize_phone_text)
train_phones['audio_path'] = train_phones['path'].map(resolve_train_audio_path)
train_phones['speaker_group'] = [infer_speaker_group(i, p) for i, p in zip(train_phones['id'], train_phones['path'])]
train_phones['has_pron_error'] = train_phones['canonical_norm'] != train_phones['transcript_norm']

baseline_df = train_phones.copy().reset_index(drop=True)
baseline_df['label_text'] = baseline_df['transcript_norm']
baseline_df['stratify_key'] = baseline_df['speaker_group'].astype(str) + '_' + baseline_df['has_pron_error'].astype(int).astype(str)

stratum_counts = baseline_df['stratify_key'].value_counts()
rare_strata = set(stratum_counts[stratum_counts < 2].index)
baseline_df.loc[baseline_df['stratify_key'].isin(rare_strata), 'stratify_key'] = baseline_df.loc[
    baseline_df['stratify_key'].isin(rare_strata), 'speaker_group'
].astype(str)

train_idx, valid_idx = train_test_split(
    np.arange(len(baseline_df)),
    test_size=0.15,
    random_state=BASELINE_SEED,
    stratify=baseline_df['stratify_key'],
)
train_df = baseline_df.iloc[train_idx].reset_index(drop=True)
valid_df = baseline_df.iloc[valid_idx].reset_index(drop=True)

print('train rows:', len(train_df))
print('valid rows:', len(valid_df))
display(train_df[['id', 'path', 'canonical_norm', 'transcript_norm']].head())


## 4. Load Public Test In CSV Order

In [ ]:
PUBLIC_TEST_PHONES_CSV = PUBLIC_TEST_ROOT / 'metadata' / 'public_test_phones.csv'
PUBLIC_TEST_AUDIO_DIR = PUBLIC_TEST_ROOT / 'audio_data' / 'public_test'

public_test_phones = pd.read_csv(PUBLIC_TEST_PHONES_CSV)
public_test_phones['canonical_norm'] = public_test_phones['canonical'].map(normalize_phone_text)

def resolve_public_test_audio_path(rel_path):
    rel = Path(str(rel_path))
    candidates = [PUBLIC_TEST_ROOT / rel, PUBLIC_TEST_AUDIO_DIR / rel.name]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

public_test_phones['audio_path'] = public_test_phones['path'].map(resolve_public_test_audio_path)
public_test_phones['audio_exists'] = public_test_phones['audio_path'].map(lambda p: p.exists())
if not bool(public_test_phones['audio_exists'].all()):
    missing = public_test_phones.loc[~public_test_phones['audio_exists'], 'path'].tolist()
    raise FileNotFoundError(f'Missing public test audio files: {missing[:10]}')

test_df = public_test_phones[['id', 'path', 'audio_path', 'canonical_norm']].copy().reset_index(drop=True)
test_df['submit_id'] = np.arange(len(test_df), dtype=np.int64)

print('public_test rows:', len(test_df))
display(test_df[['submit_id', 'id', 'path', 'canonical_norm']].head(20))


## 5. Vocabulary, Config, And Model Load

In [ ]:
transcript_phone_vocab = Counter(tok for s in train_phones['transcript_norm'] for tok in eval_phone_tokens(s))
phone_tokens_sorted = sorted(transcript_phone_vocab.keys())
blank_token = '<blank>'
unk_token = '<unk>'
id2phone = [blank_token, unk_token] + phone_tokens_sorted
phone2id = {p: i for i, p in enumerate(id2phone)}

def encode_phone_sequence(s):
    return [phone2id.get(tok, phone2id[unk_token]) for tok in eval_phone_tokens(s)]

def decode_phone_ids(ids, collapse_ctc=False):
    out = []
    prev = None
    for idx in ids:
        idx = int(idx)
        if collapse_ctc and idx == prev:
            prev = idx
            continue
        prev = idx
        if idx == phone2id[blank_token]:
            continue
        out.append(id2phone[idx] if 0 <= idx < len(id2phone) else unk_token)
    return ' '.join(out)

MODEL_NAME = 'nguyenvulebinh/wav2vec2-base-vietnamese-250h'
SAMPLING_RATE = 16000
BLANK_DECODE_PENALTY = 0.5
UNK_DECODE_PENALTY = 10.0
INFER_BATCH_SIZE = 2
POOL_FRAME_STRATEGY = 'middle'
POOL_MAX_PER_PHONE = 2000

feature_source = FINAL_MODEL_DIR if (FINAL_MODEL_DIR / 'preprocessor_config.json').exists() else MODEL_NAME
feature_extractor = AutoFeatureExtractor.from_pretrained(feature_source)

model = Wav2Vec2ForCTC.from_pretrained(
    str(FINAL_MODEL_DIR),
    vocab_size=len(id2phone),
    pad_token_id=phone2id[blank_token],
    ctc_loss_reduction='mean',
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=False,
).to(DEVICE)
model.config.pad_token_id = phone2id[blank_token]
model.config.ctc_zero_infinity = True
model.config.ctc_loss_reduction = 'mean'

print('Loaded model from:', FINAL_MODEL_DIR)
print('Feature extractor from:', feature_source)
print('Vocab size:', len(id2phone))


## 6. Inference Helpers

In [ ]:
def read_wav_pcm(path):
    with wave.open(str(path), 'rb') as wf:
        sr = wf.getframerate()
        n_channels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        frames = wf.readframes(wf.getnframes())
    if sampwidth == 2:
        data = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
    elif sampwidth == 4:
        data = np.frombuffer(frames, dtype=np.int32).astype(np.float32) / 2147483648.0
    elif sampwidth == 1:
        data = (np.frombuffer(frames, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    else:
        raise ValueError(f'Unsupported sample width: {sampwidth}')
    if n_channels > 1:
        data = data.reshape(-1, n_channels).mean(axis=1)
    return data.astype(np.float32), sr

def get_feature_lengths_from_attention(attention_mask):
    input_lengths = attention_mask.sum(-1)
    return model._get_feat_extract_output_lengths(input_lengths).to(torch.long)

def greedy_decode_one(logits, feature_length=None, blank_penalty=0.0, unk_penalty=0.0):
    adjusted = logits.copy()
    if feature_length is not None:
        adjusted = adjusted[: int(feature_length)]
    if blank_penalty:
        adjusted[:, phone2id[blank_token]] -= float(blank_penalty)
    if unk_penalty:
        adjusted[:, phone2id[unk_token]] -= float(unk_penalty)
    pred_ids = np.argmax(adjusted, axis=-1)
    return decode_phone_ids(pred_ids.tolist(), collapse_ctc=True)

def run_manual_inference(df, batch_size=INFER_BATCH_SIZE):
    model.eval()
    records = []
    for start in range(0, len(df), batch_size):
        batch_df = df.iloc[start:start + batch_size]
        waves = []
        for p in batch_df['audio_path']:
            y, sr = read_wav_pcm(p)
            if sr != SAMPLING_RATE:
                raise ValueError(f'Expected {SAMPLING_RATE} Hz, got {sr}: {p}')
            waves.append(y)
        inputs = feature_extractor(
            waves,
            sampling_rate=SAMPLING_RATE,
            padding=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True, return_dict=True)
        logits = outputs.logits.detach().cpu().numpy()
        hidden = outputs.hidden_states[-1].detach().cpu().numpy()
        feat_lens = get_feature_lengths_from_attention(inputs['attention_mask']).detach().cpu().numpy()
        for local_i, (_, row) in enumerate(batch_df.iterrows()):
            T = int(feat_lens[local_i])
            records.append({
                'id': row['id'],
                'canonical_norm': row.get('canonical_norm', None),
                'transcript_norm': row.get('transcript_norm', None),
                'logits': logits[local_i, :T].astype(np.float32),
                'hidden': hidden[local_i, :T].astype(np.float32),
                'feature_length': T,
                'raw_predict': greedy_decode_one(logits[local_i], T, BLANK_DECODE_PENALTY, UNK_DECODE_PENALTY),
            })
        if start == 0:
            print('First batch done. Example raw:', records[-len(batch_df)]['raw_predict'])
    return records


## 7. PER-MDD Helpers

In [ ]:
def align(seq1, seq2):
    GAP = -1
    MATCH = 1
    MISMATCH = -1
    n, m = len(seq1), len(seq2)
    score = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        score[i][0] = GAP * i
    for j in range(n + 1):
        score[0][j] = GAP * j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq1[j - 1] == seq2[i - 1]:
                s = MATCH
            elif seq1[j - 1] == '<eps>' or seq2[i - 1] == '<eps>':
                s = GAP
            else:
                s = MISMATCH
            score[i][j] = max(score[i - 1][j - 1] + s, score[i - 1][j] + GAP, score[i][j - 1] + GAP)
    align1, align2 = [], []
    i, j = m, n
    while i > 0 and j > 0:
        if seq1[j - 1] == seq2[i - 1]:
            s = MATCH
        elif seq1[j - 1] == '<eps>' or seq2[i - 1] == '<eps>':
            s = GAP
        else:
            s = MISMATCH
        if score[i][j] == score[i - 1][j - 1] + s:
            align1.append(seq1[j - 1]); align2.append(seq2[i - 1])
            i -= 1; j -= 1
        elif score[i][j] == score[i][j - 1] + GAP:
            align1.append(seq1[j - 1]); align2.append('<eps>')
            j -= 1
        else:
            align1.append('<eps>'); align2.append(seq2[i - 1])
            i -= 1
    while j > 0:
        align1.append(seq1[j - 1]); align2.append('<eps>'); j -= 1
    while i > 0:
        align1.append('<eps>'); align2.append(seq2[i - 1]); i -= 1
    align1.reverse(); align2.reverse()
    return align1, align2

def ops(aligned1, aligned2):
    out = []
    for r, h in zip(aligned1, aligned2):
        if r != '<eps>' and h == '<eps>':
            out.append('D')
        elif r == '<eps>' and h != '<eps>':
            out.append('I')
        elif r != h:
            out.append('S')
        else:
            out.append('C')
    return out

def align_pair(s1, s2):
    a1, a2 = align(eval_phone_tokens(s1), eval_phone_tokens(s2))
    return a1, a2, ops(a1, a2)

def transcript_correct_mask(canonical, transcript):
    ref_seq, hyp_seq, op = align_pair(canonical, transcript)
    mask = []
    for ref, hyp, o in zip(ref_seq, hyp_seq, op):
        if hyp != '<eps>':
            mask.append(o == 'C')
    return mask

def canonical_to_raw_map(canonical, raw_predict):
    ref_seq, raw_seq, _ = align_pair(canonical, raw_predict)
    mapped = []
    for ref, raw in zip(ref_seq, raw_seq):
        if ref != '<eps>':
            mapped.append(None if raw == '<eps>' else raw)
    return mapped

def log_softmax_np(x):
    x = x.astype(np.float32)
    m = np.max(x, axis=-1, keepdims=True)
    y = x - m
    return y - np.log(np.sum(np.exp(y), axis=-1, keepdims=True))

def ctc_viterbi_align(logits, target_ids, blank_id=0):
    target_ids = [int(x) for x in target_ids if int(x) != blank_id]
    T = int(logits.shape[0])
    U = len(target_ids)
    if U == 0 or T == 0:
        return None
    ext = [blank_id]
    for y in target_ids:
        ext.append(y)
        ext.append(blank_id)
    S = len(ext)
    log_probs = log_softmax_np(logits)
    neg_inf = -1e30
    dp = np.full((T, S), neg_inf, dtype=np.float32)
    bp = np.full((T, S), -1, dtype=np.int32)
    dp[0, 0] = log_probs[0, blank_id]
    if S > 1:
        dp[0, 1] = log_probs[0, ext[1]]
        bp[0, 1] = 0
    for t in range(1, T):
        for s in range(S):
            best_prev = s
            best_score = dp[t - 1, s]
            if s - 1 >= 0 and dp[t - 1, s - 1] > best_score:
                best_score = dp[t - 1, s - 1]
                best_prev = s - 1
            if s - 2 >= 0 and ext[s] != blank_id and ext[s] != ext[s - 2] and dp[t - 1, s - 2] > best_score:
                best_score = dp[t - 1, s - 2]
                best_prev = s - 2
            dp[t, s] = best_score + log_probs[t, ext[s]]
            bp[t, s] = best_prev
    end_candidates = [S - 1, S - 2]
    end_s = max(end_candidates, key=lambda s: dp[T - 1, s])
    if dp[T - 1, end_s] <= neg_inf / 2:
        return None
    path = np.zeros(T, dtype=np.int32)
    s = end_s
    for t in range(T - 1, -1, -1):
        path[t] = s
        s = bp[t, s] if t > 0 else s
        if s < 0:
            s = 0
    segments = []
    for i in range(U):
        ext_pos = 2 * i + 1
        frames = np.where(path == ext_pos)[0]
        if len(frames) == 0:
            segments.append(None)
        else:
            segments.append((int(frames[0]), int(frames[-1]) + 1))
    return segments

def segment_embedding(hidden, segment, strategy=POOL_FRAME_STRATEGY):
    if segment is None:
        return None
    s, e = segment
    if e <= s:
        return None
    if strategy == 'mean':
        return hidden[s:e].mean(axis=0)
    mid = (s + e - 1) // 2
    return hidden[mid]

def safe_encode_tokens(tokens):
    return [phone2id.get(tok, phone2id[unk_token]) for tok in tokens]

def build_reference_pool(train_records):
    by_phone = defaultdict(list)
    skipped_align = 0
    skipped_segment = 0
    for rec in train_records:
        transcript_tokens = eval_phone_tokens(rec['transcript_norm'])
        target_ids = safe_encode_tokens(transcript_tokens)
        correct_mask = transcript_correct_mask(rec['canonical_norm'], rec['transcript_norm'])
        if len(correct_mask) != len(transcript_tokens):
            skipped_align += 1
            continue
        segments = ctc_viterbi_align(rec['logits'], target_ids, phone2id[blank_token])
        if segments is None or len(segments) != len(transcript_tokens):
            skipped_align += 1
            continue
        for tok, tok_id, is_correct, seg in zip(transcript_tokens, target_ids, correct_mask, segments):
            if not is_correct or tok_id in (phone2id[blank_token], phone2id[unk_token]):
                continue
            emb = segment_embedding(rec['hidden'], seg)
            if emb is None:
                skipped_segment += 1
                continue
            by_phone[tok_id].append(emb.astype(np.float32))
    rng = np.random.default_rng(BASELINE_SEED)
    pool_embs = []
    pool_labels = []
    pool_counts = {}
    for tok_id, embs in by_phone.items():
        if len(embs) > POOL_MAX_PER_PHONE:
            idx = rng.choice(len(embs), size=POOL_MAX_PER_PHONE, replace=False)
            embs = [embs[i] for i in idx]
        pool_counts[id2phone[tok_id]] = len(embs)
        pool_embs.extend(embs)
        pool_labels.extend([tok_id] * len(embs))
    pool_embs = np.stack(pool_embs).astype(np.float32)
    norms = np.linalg.norm(pool_embs, axis=1, keepdims=True) + 1e-8
    pool_embs = pool_embs / norms
    pool_labels = np.array(pool_labels, dtype=np.int64)
    return pool_embs, pool_labels, pool_counts, {'skipped_align': skipped_align, 'skipped_segment': skipped_segment}

def retrieve_neighbors(query_emb, pool_embs, pool_labels, k):
    q = torch.tensor(query_emb, dtype=torch.float32, device=DEVICE)
    q = q / (torch.linalg.norm(q) + 1e-8)
    sims = torch.matmul(pool_embs, q)
    k_eff = min(int(k), sims.numel())
    vals, idx = torch.topk(sims, k=k_eff)
    return vals.detach().cpu().numpy(), pool_labels[idx].detach().cpu().numpy()

def vote_retrieval(query_emb, canonical_id, pool_embs, pool_labels, k, similarity_threshold, majority_ratio, margin):
    sims, labels = retrieve_neighbors(query_emb, pool_embs, pool_labels, k)
    keep = sims >= similarity_threshold
    if not np.any(keep):
        return {
            'decision_id': canonical_id,
            'decision': 'fallback_no_neighbor',
            'top_phone': id2phone[canonical_id],
            'top_similarity': float(sims[0]) if len(sims) else 0.0,
            'majority_ratio': 0.0,
            'canonical_best_similarity': 0.0,
        }
    filt_sims = sims[keep]
    filt_labels = labels[keep]
    label_counts = Counter(filt_labels.tolist())
    top_label, top_count = label_counts.most_common(1)[0]
    ratio = top_count / len(filt_labels)
    top_sim = float(np.max(filt_sims[filt_labels == top_label]))
    canonical_sims = filt_sims[filt_labels == canonical_id]
    canonical_best = float(np.max(canonical_sims)) if len(canonical_sims) else -1.0
    if top_label != canonical_id and ratio >= majority_ratio and top_sim >= similarity_threshold and (top_sim - canonical_best) >= margin:
        return {
            'decision_id': int(top_label),
            'decision': 'retrieval_change',
            'top_phone': id2phone[int(top_label)],
            'top_similarity': top_sim,
            'majority_ratio': float(ratio),
            'canonical_best_similarity': canonical_best,
        }
    return {
        'decision_id': canonical_id,
        'decision': 'fallback_canonical',
        'top_phone': id2phone[int(top_label)],
        'top_similarity': top_sim,
        'majority_ratio': float(ratio),
        'canonical_best_similarity': canonical_best,
    }

def correct_record(rec, pool_embs, pool_labels, k=5, similarity_threshold=0.75, majority_ratio=0.8, margin=0.06, require_raw_agreement=True):
    canonical_tokens = eval_phone_tokens(rec['canonical_norm'])
    canonical_ids = safe_encode_tokens(canonical_tokens)
    raw_map = canonical_to_raw_map(rec['canonical_norm'], rec['raw_predict'])
    segments = ctc_viterbi_align(rec['logits'], canonical_ids, phone2id[blank_token])
    if segments is None or len(segments) != len(canonical_tokens):
        return rec['canonical_norm'], [{'decision': 'fallback_alignment_failed'}]
    corrected_ids = []
    debug_rows = []
    for pos, (tok, tok_id, seg) in enumerate(zip(canonical_tokens, canonical_ids, segments)):
        emb = segment_embedding(rec['hidden'], seg)
        raw_phone = raw_map[pos] if pos < len(raw_map) else None
        if emb is None or tok_id == phone2id[unk_token]:
            corrected_ids.append(tok_id)
            debug_rows.append({'pos': pos, 'canonical_phone': tok, 'raw_phone': raw_phone, 'decision': 'fallback_no_embedding'})
            continue
        vote = vote_retrieval(emb, tok_id, pool_embs, pool_labels, k, similarity_threshold, majority_ratio, margin)
        decision_id = vote['decision_id']
        if require_raw_agreement and decision_id != tok_id:
            raw_id = phone2id.get(raw_phone, None) if raw_phone is not None else None
            if raw_id != decision_id:
                vote['decision'] = 'fallback_raw_disagrees'
                decision_id = tok_id
        corrected_ids.append(decision_id)
        debug_rows.append({
            'pos': pos,
            'canonical_phone': tok,
            'raw_phone': raw_phone,
            'decision_phone': id2phone[int(decision_id)],
            **vote,
        })
    return ' '.join(id2phone[int(i)] for i in corrected_ids if int(i) != phone2id[blank_token]), debug_rows


## 8. Load Best Config, Rebuild Pool, And Predict

In [ ]:
best_config_path = PER_MDD_DIR / 'best_per_mdd_config.json'
if best_config_path.exists():
    best_cfg = json.loads(best_config_path.read_text(encoding='utf-8'))
    print('Loaded best config from:', best_config_path)
else:
    best_cfg = {
        'k': 5,
        'similarity_threshold': 0.75,
        'majority_ratio': 0.8,
        'margin': 0.06,
        'require_raw_agreement': True,
    }
    print('best_per_mdd_config.json not found. Using fallback config:', best_cfg)

train_cache = run_manual_inference(train_df, batch_size=INFER_BATCH_SIZE)
print('Cached train rows:', len(train_cache))

pool_embs_np, pool_labels_np, pool_counts, pool_stats = build_reference_pool(train_cache)
pool_embs = torch.tensor(pool_embs_np, dtype=torch.float32, device=DEVICE)
pool_labels = torch.tensor(pool_labels_np, dtype=torch.long, device=DEVICE)

print('Pool embeddings:', pool_embs.shape)
print('Unique pool phones:', len(pool_counts))
print('Pool stats:', pool_stats)

test_cache = run_manual_inference(test_df, batch_size=INFER_BATCH_SIZE)
print('Cached test rows:', len(test_cache))

predictions = []
debug_rows = []
for rec, (_, row) in zip(test_cache, test_df.iterrows()):
    pred, dbg = correct_record(rec, pool_embs, pool_labels, **best_cfg)
    predictions.append(pred)
    for d in dbg:
        debug_rows.append({
            'submit_id': int(row['submit_id']),
            'source_id': row['id'],
            'path': row['path'],
            **d,
        })


## 9. Save Results

In [ ]:
results_df = pd.DataFrame({'predict': predictions})
predictions_df = pd.DataFrame({
    'id': test_df['submit_id'].astype(int),
    'predict': predictions,
})

results_path = PER_MDD_DIR / 'results.csv'
predictions_path = PER_MDD_DIR / 'predictions.csv'
debug_path = PER_MDD_DIR / 'public_test_predict_only_debug.csv'
order_debug_path = PER_MDD_DIR / 'public_test_csv_order_debug.csv'

results_df.to_csv(results_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
pd.DataFrame(debug_rows).to_csv(debug_path, index=False)
test_df[['submit_id', 'id', 'path']].assign(predict=predictions).to_csv(order_debug_path, index=False)

print('Wrote:', results_path)
print('Wrote:', predictions_path)
print('Wrote:', debug_path)
print('Wrote:', order_debug_path)
display(predictions_df.head(20))
